# MVPA — 01: Compute Temporal Decoding

Runs temporal decoding (and optionally temporal generalization and cross-decoding)
for one or more condition pairs.

**Warning:** this is computationally expensive. Use `n_jobs=-1` and start with
fewer repeats (`n_repeats=10`) to test before running the full analysis.

**Output:** per-subject decoding scores saved to the analysis directory

In [ ]:
%load_ext autoreload
%autoreload 2

from eeg_toolkit import load_config, decode_all, load_all_scores

# ── Update this path ──
cfg = load_config('../../configs/your_experiment.yaml')

print("Setup OK")

In [ ]:
# ── Define conditions to decode ──
# Keys = condition labels (used for output filenames and plots)
# Values = list of event codes belonging to that condition
# ── Update event codes to match your experiment ──
conditions = {
    'condition_a': [10, 11, 12],   # replace with your event codes
    'condition_b': [20, 21, 22],
}

summary, group = decode_all(
    cfg,
    window_name='your_window',        # must match an epoching window name
    conditions=conditions,
    analysis_name='a_vs_b',           # used in output filenames
    classifier='linear_svc',
    n_folds=3,
    n_repeats=100,                    # reduce for testing
    resample_sfreq=256,               # set to None to skip resampling
    baseline=(-0.2, 0.0),             # set to None to skip baseline
    n_null_permutations=1,            # increase for publication (e.g. 100)
    compute_distances=True,
    n_jobs=-1,
    overwrite=False,
)

In [ ]:
# ── Optional: temporal generalization matrix ──
# Trains a classifier at each time point and tests it at all other time points.
# Slower — reduce resample_sfreq (e.g. 128) and n_repeats (e.g. 1).
summary_tg, group_tg = decode_all(
    cfg,
    window_name='your_window',
    conditions=conditions,
    analysis_name='a_vs_b_tg',
    classifier='linear_svc',
    n_folds=3,
    n_repeats=1,
    resample_sfreq=128,
    baseline=(-0.2, 0.0),
    compute_temporal_gen=True,
    n_jobs=-1,
    overwrite=False,
)

In [ ]:
# ── Optional: cross-decoding ──
# Train on one set of conditions, test on another.
from eeg_toolkit.mvpa import cross_decode_all

train_conditions = {
    'left':  [10, 11],
    'right': [12, 13],
}
test_conditions = {
    'left':  [20, 21],
    'right': [22, 23],
}

summary_xdec, group_xdec = cross_decode_all(
    cfg,
    window_name='your_window',
    train_conditions=train_conditions,
    test_conditions=test_conditions,
    analysis_name='cross_decode',
    classifier='linear_svc',
    resample_sfreq=256,
    baseline=(-0.2, 0.0),
    bidirectional=True,
    n_null_permutations=1,
    n_jobs=-1,
    overwrite=False,
)